# Group 3 — Bonus: a Chainlink oracle resolves the market

**Problem in the base version:** the market creator decides the result. A dishonest creator could pick the wrong outcome.

**Bonus:** `OracleResolver.sol` creates a price market on our existing `TobesureMarket` contract and lets a **Chainlink price feed** decide it:

1. The resolver contract creates the market, so the resolver itself is the market creator.
2. Question: *Will ETH/USD be above $X at time T?* The threshold X and time T are stored on-chain.
3. After T, **anyone** can call `resolve()`. The contract reads the Chainlink price, checks that it is valid and fresh, and resolves YES if price > X, otherwise NO.
4. Payouts run exactly as before, through the unchanged TobesureMarket contract.

Needs in the same folder: `keys.env` (with `CONTRACT_ADDRESS` set), `TobesureMarket.sol`, `OracleResolver.sol`.

## 1 · Setup and connection

In [ ]:
import os, time
from datetime import datetime, timezone
import pandas as pd
from dotenv import load_dotenv
from web3 import Web3
from web3.logs import DISCARD
from eth_account import Account
import solcx

load_dotenv("keys.env") or load_dotenv(".env")
NETWORK_MODE = os.getenv("NETWORK_MODE", "sepolia").lower()
EXPLORER = "https://sepolia.etherscan.io"
SOLC_VERSION = "0.8.20"

def load_key(name):
    k = (os.getenv(name) or "").strip().strip('"').strip("'")
    if not k:
        return None
    return k if k.startswith("0x") else "0x" + k

if NETWORK_MODE == "sepolia":
    w3 = Web3(Web3.HTTPProvider(f"https://sepolia.infura.io/v3/{(os.getenv('INFURA_API_KEY') or '').strip()}"))
    deployer = Account.from_key(load_key("DEPLOYER_PRIVATE_KEY"))
    members = [{"name": os.getenv(f"MEMBER_{i}_NAME", f"Member {i}"), "acct": Account.from_key(load_key(f"MEMBER_{i}_PRIVATE_KEY"))}
               for i in range(1, 9) if load_key(f"MEMBER_{i}_PRIVATE_KEY")]
else:
    from web3 import EthereumTesterProvider
    w3 = Web3(EthereumTesterProvider())
    deployer = Account.create()
    w3.eth.send_transaction({"from": w3.eth.accounts[0], "to": deployer.address, "value": w3.to_wei(10, "ether")})
    members = [{"name": n, "acct": Account.create()} for n in ["Max", "Ben", "Ale", "Sarita", "Anthony"]]
    for m in members:
        w3.eth.send_transaction({"from": w3.eth.accounts[0], "to": m["acct"].address, "value": w3.to_wei(1, "ether")})

assert w3.is_connected(), "Not connected: check INFURA_API_KEY in keys.env"
CHAIN_ID = w3.eth.chain_id
eth = lambda wei: float(w3.from_wei(wei, "ether"))
print("Mode:", NETWORK_MODE, "| chain id:", CHAIN_ID)
print("Platform:", deployer.address, "|", round(eth(w3.eth.get_balance(deployer.address)), 6), "ETH")
print("Members:", ", ".join(m["name"] for m in members))

## 2 · Compile both contracts

In [ ]:
MOCK_FEED = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;
contract MockV3Aggregator {
    uint8 public decimals = 8; string public description = "ETH / USD (mock)";
    int256 public answer; uint256 public updatedAt; uint80 public round;
    constructor(int256 a) { set(a); }
    function set(int256 a) public { answer = a; updatedAt = block.timestamp; round++; }
    function latestRoundData() external view returns (uint80, int256, uint256, uint256, uint80) {
        return (round, answer, updatedAt, updatedAt, round);
    }
}
"""
def read(path):
    with open(path) as f:
        return f.read()
sources = {
    "TobesureMarket.sol": {"content": read("TobesureMarket.sol")},
    "OracleResolver.sol": {"content": read("OracleResolver.sol")},
    "Mock.sol": {"content": MOCK_FEED},
}
solcx.install_solc(SOLC_VERSION)
out = solcx.compile_standard({"language": "Solidity", "sources": sources,
        "settings": {"optimizer": {"enabled": True, "runs": 200},
                     "outputSelection": {"*": {"*": ["abi", "evm.bytecode.object"]}}}}, solc_version=SOLC_VERSION)
art = lambda f, c: (out["contracts"][f][c]["abi"], out["contracts"][f][c]["evm"]["bytecode"]["object"])
MARKET_ABI, MARKET_BIN = art("TobesureMarket.sol", "TobesureMarket")
ORACLE_ABI, ORACLE_BIN = art("OracleResolver.sol", "OracleResolver")
MOCK_ABI, MOCK_BIN = art("Mock.sol", "MockV3Aggregator")
FEED_ABI = [x for x in MOCK_ABI if x.get("name") in ("decimals", "description", "latestRoundData")]
print("Compiled ✓  OracleResolver functions:", sorted(x["name"] for x in ORACLE_ABI if x["type"] == "function"))

## 3 · Transaction helpers (gas estimate + status check)

In [ ]:
tx_log = []

def send_tx(fn, account, value_eth=0, label=""):
    params = {"from": account.address, "nonce": w3.eth.get_transaction_count(account.address, "pending"),
              "value": w3.to_wei(value_eth, "ether"), "chainId": CHAIN_ID}
    tx = fn.build_transaction(params)
    tx["gas"] = int(w3.eth.estimate_gas(tx) * 1.3)
    signed = account.sign_transaction(tx)
    raw = getattr(signed, "raw_transaction", None) or getattr(signed, "rawTransaction")
    h = w3.eth.send_raw_transaction(raw)
    r = w3.eth.wait_for_transaction_receipt(h, timeout=300)
    hx = "0x" + h.hex().removeprefix("0x")
    print(("✅" if r.status == 1 else "❌ REVERTED"), label, f"| gas {r.gasUsed:,} | block {r.blockNumber}")
    if NETWORK_MODE == "sepolia":
        print("   ", f"{EXPLORER}/tx/{hx}")
    tx_log.append({"step": label, "tx_hash": hx, "block": r.blockNumber, "gas_used": r.gasUsed, "status": r.status,
                   "etherscan": f"{EXPLORER}/tx/{hx}" if NETWORK_MODE == "sepolia" else "(local)"})
    assert r.status == 1, f"Transaction failed: {label}"
    return r

def top_up(to, amount_eth, label):
    tx = {"to": to, "value": w3.to_wei(amount_eth, "ether"), "chainId": CHAIN_ID,
          "nonce": w3.eth.get_transaction_count(deployer.address, "pending"),
          "maxPriorityFeePerGas": w3.to_wei(1, "gwei"), "maxFeePerGas": max(w3.eth.gas_price * 2, w3.to_wei(2, "gwei"))}
    tx["gas"] = int(w3.eth.estimate_gas({**tx, "from": deployer.address}) * 1.5)
    signed = deployer.sign_transaction(tx)
    raw = getattr(signed, "raw_transaction", None) or getattr(signed, "rawTransaction")
    r = w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(raw), timeout=300)
    print("   ", "✅" if r.status == 1 else "❌", label)

def wait_for_balance(addr, minimum_eth, timeout=120):
    start = time.time()
    while True:
        bal = eth(w3.eth.get_balance(addr))
        if bal >= minimum_eth or time.time() - start > timeout:
            return bal
        time.sleep(5)
print("Helpers ready ✓")

## 4 · Our existing market contract and the Chainlink price feed

Uses `CONTRACT_ADDRESS` from keys.env (the TobesureMarket we already deployed). The notebook checks the feed **on-chain** before using it:
it tries ETH/USD first and falls back to BTC/USD (the feed Chainlink's own Sepolia guide uses).

In [ ]:
FEED_CANDIDATES = [
    ("ETH/USD", os.getenv("PRICE_FEED_ADDRESS") or "0x694AA1769357215DE4FAC081bf1f309aDC325306"),
    ("BTC/USD", "0x1b44F3514812d835EB1BDB0acB33d3fA3351Ee43"),
]

if NETWORK_MODE == "sepolia":
    MARKET_ADDRESS = Web3.to_checksum_address(os.getenv("CONTRACT_ADDRESS").strip())
    feed = None
    for pair, addr in FEED_CANDIDATES:
        try:
            c = w3.eth.contract(address=Web3.to_checksum_address(addr), abi=FEED_ABI)
            desc = c.functions.description().call()
            rd = c.functions.latestRoundData().call()
            if rd[1] > 0:
                feed, PAIR = c, desc
                break
        except Exception as e:
            print(f"{pair} feed not usable at {addr}: {str(e)[:80]}")
    assert feed is not None, "No working Chainlink feed found; tell Claude"
else:
    r = send_tx(w3.eth.contract(abi=MARKET_ABI, bytecode=MARKET_BIN).constructor(200, 100), deployer, label="Deploy TobesureMarket (local)")
    MARKET_ADDRESS = r.contractAddress
    r = send_tx(w3.eth.contract(abi=MOCK_ABI, bytecode=MOCK_BIN).constructor(3500 * 10**8), deployer, label="Deploy mock price feed (local)")
    feed = w3.eth.contract(address=r.contractAddress, abi=FEED_ABI)
    PAIR = feed.functions.description().call()

market = w3.eth.contract(address=MARKET_ADDRESS, abi=MARKET_ABI)
DECIMALS = feed.functions.decimals().call()
_, answer, _, updated_at, _ = feed.functions.latestRoundData().call()
PRICE = answer / 10**DECIMALS
print("Market contract:", MARKET_ADDRESS)
print(f"Chainlink feed: {PAIR} at {feed.address}")
print(f"Current price: ${PRICE:,.2f}  (updated {datetime.fromtimestamp(updated_at, tz=timezone.utc):%Y-%m-%d %H:%M UTC}, {DECIMALS} decimals)")

## 5 · Deploy the OracleResolver (or reuse it)

`maxStaleness` = 24 hours: testnet feeds update less often than mainnet. On mainnet you would use the feed's heartbeat (often 1 hour).
After the first deployment, paste the address into keys.env as `ORACLE_ADDRESS=...`.

In [ ]:
MAX_STALENESS = 24 * 3600
existing = (os.getenv("ORACLE_ADDRESS") or "").strip() if NETWORK_MODE == "sepolia" else ""
if existing:
    ORACLE_ADDRESS = Web3.to_checksum_address(existing)
    print("Reusing OracleResolver:", ORACLE_ADDRESS)
else:
    r = send_tx(w3.eth.contract(abi=ORACLE_ABI, bytecode=ORACLE_BIN).constructor(MARKET_ADDRESS, feed.address, MAX_STALENESS),
                deployer, label="Deploy OracleResolver")
    ORACLE_ADDRESS = r.contractAddress
    print("\n🔮 OracleResolver deployed at:", ORACLE_ADDRESS)
    if NETWORK_MODE == "sepolia":
        print("   ", f"{EXPLORER}/address/{ORACLE_ADDRESS}")
        print("   → paste into keys.env as ORACLE_ADDRESS=... so later runs reuse it")
oracle = w3.eth.contract(address=ORACLE_ADDRESS, abi=ORACLE_ABI)
p, u = oracle.functions.latestPrice().call()
print(f"Oracle sees: ${p / 10**DECIMALS:,.2f} (checks passed: valid, complete round, not stale)")

## 6 · Create the oracle market

The threshold is the current price rounded to the nearest dollar, so the result depends on where the price moves.
`ORACLE_WINDOW` = how long betting stays open before the oracle can resolve (4 minutes for the demo).

In [ ]:
ORACLE_WINDOW = 4 * 60
_, answer, _, _, _ = feed.functions.latestRoundData().call()
THRESHOLD_USD = round(answer / 10**DECIMALS)
THRESHOLD = THRESHOLD_USD * 10**DECIMALS
close_at = datetime.fromtimestamp(w3.eth.get_block("latest").timestamp + ORACLE_WINDOW, tz=timezone.utc)
asset = PAIR.split("/")[0].strip().split(" ")[0]
QUESTION = f"Will {asset}/USD be above ${THRESHOLD_USD:,} at {close_at:%H:%M} UTC on {close_at:%d %b %Y}? (Chainlink)"

r = send_tx(oracle.functions.createPriceMarket(QUESTION, THRESHOLD, ORACLE_WINDOW), deployer, label="Create oracle market")
ev = oracle.events.OracleMarketCreated().process_receipt(r, errors=DISCARD)[0]["args"]
OMARKET_ID = ev["marketId"]
RESOLVE_AFTER = ev["resolveAfter"]
print(f"\n🆕 Market #{OMARKET_ID}: {QUESTION}")
print("   Market creator on TobesureMarket:", market.functions.getMarket(OMARKET_ID).call()[1], "(= the OracleResolver contract)")

## 7 · Bets on the oracle market

In [ ]:
ORACLE_BETS = [   # (side, amount) in member order: Max, Ben, Ale, Sarita, Anthony. True = YES (above), False = NO
    (True,  0.001),
    (False, 0.001),
    (True,  0.001),
    (False, 0.001),
    (False, 0.001),
]
for m, (side, amount) in zip(members, ORACLE_BETS):
    addr = m["acct"].address
    if eth(w3.eth.get_balance(addr)) < amount + 0.001:
        top_up(addr, 0.003, f"Top-up {m['name']}")
    if wait_for_balance(addr, amount + 0.0005) < amount + 0.0005:
        print(f"⚠️ {m['name']} has too little ETH, bet skipped"); continue
    send_tx(market.functions.placeBet(OMARKET_ID, side), m["acct"], value_eth=amount,
            label=f"Oracle bet {'YES' if side else 'NO'} {amount} ETH — {m['name']}")

q, creator, deadline, yes_pool, no_pool, outcome, resolved = market.functions.getMarket(OMARKET_ID).call()
print(f"\nYES (above ${THRESHOLD_USD:,}) {eth(yes_pool):.4f} ETH | NO {eth(no_pool):.4f} ETH | implied P(YES) {market.functions.yesProbabilityBps(OMARKET_ID).call()/100:.1f}%")

## 8 · Wait for the deadline

In [ ]:
if NETWORK_MODE == "local":
    w3.provider.ethereum_tester.time_travel(RESOLVE_AFTER + 1); w3.provider.ethereum_tester.mine_blocks(1)
while True:
    left = RESOLVE_AFTER - w3.eth.get_block("latest").timestamp
    if left <= 0:
        break
    print(f"⏳ {left} s until the oracle may resolve…", end="\r")
    time.sleep(min(15, max(left, 1)))
print("\n✅ Deadline passed: betting is closed, resolution is open to anyone.")

## 9 · 🎤 Resolve with Chainlink, triggered by a normal member

Ale calls `resolve()`. Ale is **not** the creator and **not** the owner, yet the call works, because the outcome comes from the oracle, not from the caller.

In [ ]:
TRIGGER = next(m for m in members if m["name"] == "Ale") if any(m["name"] == "Ale" for m in members) else members[0]
watch = {m["name"]: m["acct"].address for m in members}
before = {k: w3.eth.get_balance(a) for k, a in watch.items()}

r = send_tx(oracle.functions.resolve(OMARKET_ID), TRIGGER["acct"], label=f"Oracle resolve #{OMARKET_ID} (by {TRIGGER['name']})")
ev = oracle.events.OracleResolved().process_receipt(r, errors=DISCARD)[0]["args"]
price = ev["price"] / 10**DECIMALS
print(f"\n🔮 Chainlink {PAIR}: ${price:,.2f} (updated {datetime.fromtimestamp(ev['priceUpdatedAt'], tz=timezone.utc):%H:%M:%S UTC})")
print(f"   Threshold ${THRESHOLD_USD:,} → {'YES wins' if ev['yesWins'] else 'NO wins'} | triggered by {TRIGGER['name']} ({ev['triggeredBy'][:10]}…)")

pays = market.events.Payout().process_receipt(r, errors=DISCARD)
fees = market.events.FeePaid().process_receipt(r, errors=DISCARD)
name_of = {v: k for k, v in watch.items()}
name_of[ORACLE_ADDRESS] = "OracleResolver (creator fee)"
name_of[market.functions.owner().call()] = "Platform"
rows = [{"recipient": name_of.get(e["args"]["to"], e["args"]["to"][:10]), "type": "winnings / refund", "ETH": eth(e["args"]["amount"])} for e in pays]
rows += [{"recipient": name_of.get(e["args"]["to"], e["args"]["to"][:10]), "type": "platform fee" if e["args"]["isPlatformFee"] else "creator fee", "ETH": eth(e["args"]["amount"])} for e in fees]
display(pd.DataFrame(rows))
print("Contract balance after:", eth(w3.eth.get_balance(MARKET_ADDRESS)), "ETH")

## 10 · Security checks (expected to fail ✋)

In [ ]:
def expect_revert(desc, fn, sender):
    try:
        fn.call({"from": sender}); print(f"❌ {desc}: did NOT revert")
    except Exception as e:
        print(f"✅ {desc}: reverted → {str(e)[:70]}")

someone = members[-1]["acct"].address
expect_revert("Resolve the same oracle market twice", oracle.functions.resolve(OMARKET_ID), someone)
expect_revert("A member resolves the market directly", market.functions.resolveMarket(OMARKET_ID, 1), someone)
expect_revert("A member creates an oracle market", oracle.functions.createPriceMarket("x", 1, 60), someone)
expect_revert("Resolve a market that is not an oracle market", oracle.functions.resolve(999), someone)

## 11 · Transaction log

In [ ]:
log = pd.DataFrame(tx_log); log.to_csv("oracle_tx_log.csv", index=False)
print("OracleResolver:", f"{EXPLORER}/address/{ORACLE_ADDRESS}" if NETWORK_MODE == "sepolia" else ORACLE_ADDRESS)
for t in tx_log:
    print(f"- {t['step']}: {t['etherscan']}")
log[["step", "tx_hash", "block", "gas_used", "status"]]